# Paso D — Modelo Predictivo Primer Año → GO-i
**Proyecto:** Saving Brains - KMC 20-Year Follow-up

**Objetivo:** Predecir el fenotipo cognitivo GO-i (GO-1 Alto / GO-2 Bajo) a los 20 años
usando variables clínicas del período neonatal y primer año de vida.

**Pipeline:**
1. Carga y definición de variables por fase temporal
2. Modelos incrementales XGBoost (M1→M4) con MICE dentro de CV
3. Interpretabilidad SHAP sobre M4
4. Figuras
5. Resumen y export

---

In [0]:
"""
Project : Saving Brains — KMC 20-Year Follow-up
Module  : Paso D — Modelo predictivo primer año → GO-i
          Clasificación binaria incremental con XGBoost + SHAP

MUESTRA
───────
G1 = prematuros (preterm==1, KMC+TC), n=386
GO-1 Alto: 273 (70.7%)  |  GO-2 Bajo: 113 (29.3%)

DISEÑO INCREMENTAL (4 modelos)
───────────────────────────────
M1: Embarazo + Nacimiento          (F1+F2,  29 vars)
M2: + Neonatal + Socioeconómico    (F1-F4,  61 vars)
M3: + 40 semanas + Psicomotor      (F1-F6,  93 vars)
M4: + HOMET + Seguimiento 12m      (F1-F9, 181 vars) ← modelo completo

MODELO
──────
XGBClassifier con calibración isotónica
  - CV: StratifiedKFold(5), out-of-fold predictions
  - MICE (IterativeImputer BayesianRidge) DENTRO de cada fold → sin leakage
  - Umbral de clasificación: optimizado por F1 en CV

EVALUACIÓN
──────────
  AUC-ROC | Brier score | F1 | Sensibilidad | Especificidad

INTERPRETABILIDAD
─────────────────
  SHAP TreeExplainer sobre M4 completo — Top 20 variables
"""

## Importaciones

In [0]:
%pip install shap

In [0]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import io, sys
_stderr_orig = sys.stderr
sys.stderr = open(os.devnull, 'w')

import logging
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore')

sys.stderr.close()
sys.stderr = _stderr_orig
sns.set_theme(style='whitegrid')
print("Imports OK ✓")

## Configuración

In [0]:
INPUT_MASTER   = '/Volumes/workspace/default/kmc_data/kmc_dataset_procesado_completo.csv'
INPUT_CLUSTERS = '/Volumes/workspace/default/kmc_data/clusters_GOiv5.csv'
OUT_METRICAS   = 'paso_d_resultados.csv'
OUT_SHAP       = 'paso_d_shap_importancias.csv'
OUT_MODELO     = 'paso_d_modelo_m4.joblib'
LOG_FILE       = 'paso_d.log'

RANDOM_STATE   = 42
N_FOLDS        = 5

# ── Variables excluidas (outcomes 20 años — anti-leakage) ─────────────────
LEAK_PREFIXES = [
    'WASI_','TAP_','CVLT_','VMI_',
    'ICFES_','METP_','ABCL_','ABCLP_','ABCLBF_','ASR_',
    'RT_','NHPT_','SURV24_','SURVEY24_',
    'Years15_','Kidscreen',
    'AUDIO_','OPHT_','PEDIATRIC_',
    'CONNERS_',
    'ASEG_','TRAC_','APARC_','BROADMAN_','GMPARC_','CAM_','ROI_',
    'FOLLOW_',
    'LIFEHABITS_',
    'EX41_APEGO',
    'LEARN_','LATERALITY_','AUTOESTIMA_','CES_D_','APEGO20_',
    'DOMICILIARY_',
]

# ── Fases temporales ───────────────────────────────────────────────────────
FASES = {
    'F1_embarazo'   : ['PRE_'],
    'F2_nacimiento' : ['BIRTH_'],
    'F3_neonatal'   : ['NEO_'],
    'F4_socioec'    : ['SCB_','CSP_'],
    'F5_40semanas'  : ['EX41_'],
    'F6_psicomotor' : ['PMD_'],
    'F7_homet'      : ['HOMET_'],
    'F8_seguim12m'  : ['FOLL12M_'],
    'F9_nutricion'  : ['NUT12M_'],
}

# ── Modelos incrementales ──────────────────────────────────────────────────
MODELOS_INC = {
    'M1_embarazo_nacimiento': ['F1_embarazo','F2_nacimiento'],
    'M2_neonatal_socioec'   : ['F1_embarazo','F2_nacimiento',
                               'F3_neonatal','F4_socioec'],
    'M3_40sem_psicomotor'   : ['F1_embarazo','F2_nacimiento',
                               'F3_neonatal','F4_socioec',
                               'F5_40semanas','F6_psicomotor'],
    'M4_completo'           : ['F1_embarazo','F2_nacimiento',
                               'F3_neonatal','F4_socioec',
                               'F5_40semanas','F6_psicomotor',
                               'F7_homet','F8_seguim12m','F9_nutricion'],
}

# ── Hiperparámetros XGBoost ────────────────────────────────────────────────
XGB_PARAMS = dict(
    objective        = 'binary:logistic',
    eval_metric      = 'auc',
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,    # regularización para GO-2 n=113
    reg_lambda       = 2.0,
    scale_pos_weight = 1,    # desbalance moderado (70/30)
    random_state     = RANDOM_STATE,
    n_jobs           = -1,
    verbosity        = 0,
)

COLORES_MOD = {
    'M1_embarazo_nacimiento': '#AED6F1',
    'M2_neonatal_socioec'   : '#5DADE2',
    'M3_40sem_psicomotor'   : '#2E86C1',
    'M4_completo'           : '#1A5276',
}
LABELS = {
    'M1_embarazo_nacimiento': 'M1 — Embarazo + Nacimiento',
    'M2_neonatal_socioec'   : 'M2 — + Neonatal + Socioeconómico',
    'M3_40sem_psicomotor'   : 'M3 — + 40 sem + Psicomotor',
    'M4_completo'           : 'M4 — Completo (12 meses)',
}

## Configuración de Logging

In [0]:
_fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
_fh  = logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8')
_fh.setFormatter(_fmt)
_stream = io.TextIOWrapper(
    sys.stdout.buffer, encoding='utf-8', errors='replace', line_buffering=True
) if hasattr(sys.stdout, 'buffer') else sys.stdout
_ch = logging.StreamHandler(_stream)
_ch.setFormatter(_fmt)
logging.basicConfig(level=logging.INFO, handlers=[_fh, _ch])

def sep(title='', width=76):
    t = title.strip()
    if t:
        logging.info(f"\n{'─'*4} {t} {'─'*max(2, width-len(t)-6)}")
    else:
        logging.info('─' * width)

def to_num(s):
    return pd.to_numeric(s, errors='coerce')

## Paso 0 — Carga y Definición de Variables

In [0]:
def paso_0_carga(master_path, clusters_path):
    sep('PASO 0 — CARGA Y DEFINICIÓN DE VARIABLES')

    df  = pd.read_csv(master_path, low_memory=False)
    cl  = pd.read_csv(clusters_path)
    df  = df.merge(cl[['code','GO_i']], on='code', how='inner')
    logging.info(f"Dataset completo: {df.shape[0]} × {df.shape[1]}")

    # Muestra G1 — prematuros KMC + TC
    grupo = to_num(df['ubicac'])
    pret  = to_num(df['preterm'])
    mask  = grupo.isin([1,2]) & (pret == 1) & df['GO_i'].notna()
    df_g1 = df[mask].copy().reset_index(drop=True)
    logging.info(f"Muestra G1 (prematuros, KMC+TC): n={len(df_g1)}")
    for go, label in [(1,'GO-1 Alto'),(2,'GO-2 Bajo')]:
        n = (df_g1['GO_i']==go).sum()
        logging.info(f"  {label}: n={n} ({n/len(df_g1)*100:.1f}%)")

    # Inventario de variables por fase (sin leakage)
    vars_por_fase = {}
    sep('Inventario de variables por fase (sin leakage)')
    logging.info(f"  {'Fase':<22} {'N vars':>7}  {'Miss %':>8}  {'Completo %':>12}")
    logging.info('  '+ '─'*52)

    for fase, prefijos in FASES.items():
        cols = []
        for c in df_g1.columns:
            if not any(c.startswith(p) for p in prefijos):
                continue
            if any(c.startswith(L) for L in LEAK_PREFIXES):
                continue
            s = to_num(df_g1[c])
            if s.nunique() < 2:
                continue
            cols.append(c)

        vars_por_fase[fase] = cols
        if cols:
            X_f  = df_g1[cols].apply(to_num)
            miss = X_f.isna().mean().mean()*100
            comp = X_f.notna().all(axis=1).mean()*100
            logging.info(f"  {fase:<22} {len(cols):>7}  {miss:>7.1f}%  {comp:>11.1f}%")
        else:
            logging.info(f"  {fase:<22}       0  (vacía)")

    # Target binario: GO-2 Bajo = 1 (positivo = fenotipo de riesgo)
    y = (df_g1['GO_i'] == 2).astype(int)
    logging.info(f"\n  Target: GO-2 Bajo=1 (n={y.sum()})  GO-1 Alto=0 (n={(~y.astype(bool)).sum()})")
    logging.info(f"  Prevalencia GO-2: {y.mean()*100:.1f}%")

    return df_g1, y, vars_por_fase


df_g1, y, vars_por_fase = paso_0_carga(INPUT_MASTER, INPUT_CLUSTERS)

## Paso 1 — Funciones de Evaluación con MICE dentro de CV

In [0]:
def mice_impute_fold(X_train, X_test):
    """
    MICE ajustado SOLO sobre train, aplicado a train+test.
    Evita leakage entre folds de CV.
    """
    mice = IterativeImputer(
        estimator    = BayesianRidge(),
        max_iter     = 10,
        random_state = RANDOM_STATE,
        verbose      = 0,
    )
    X_tr_imp = mice.fit_transform(X_train)
    X_te_imp = mice.transform(X_test)
    return X_tr_imp, X_te_imp, mice


def evaluar_modelo(X, y, nombre, df_g1_idx):
    """
    Evaluación completa con StratifiedKFold(5) + MICE dentro de CV.
    Retorna: métricas, oof_probs, oof_preds, bundle (modelo+preprocesador).
    """
    X_arr = X.apply(to_num).values.astype(float)
    y_arr = y.values

    skf      = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)
    oof_prob = np.zeros(len(y_arr))
    aucs, briers = [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X_arr, y_arr), 1):
        X_tr, X_te = X_arr[tr_idx], X_arr[te_idx]
        y_tr, y_te = y_arr[tr_idx], y_arr[te_idx]

        # MICE dentro del fold — sin leakage
        X_tr_imp, X_te_imp, _ = mice_impute_fold(X_tr, X_te)

        # Escalar
        scaler   = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_tr_imp)
        X_te_sc  = scaler.transform(X_te_imp)

        # Modelo
        clf = XGBClassifier(**XGB_PARAMS)
        clf.fit(X_tr_sc, y_tr,
                eval_set=[(X_te_sc, y_te)],
                verbose=False)

        probs = clf.predict_proba(X_te_sc)[:, 1]
        oof_prob[te_idx] = probs
        aucs.append(roc_auc_score(y_te, probs))
        briers.append(brier_score_loss(y_te, probs))

    # Umbral óptimo por F1 sobre OOF completo
    thresholds = np.linspace(0.1, 0.9, 81)
    f1s        = [f1_score(y_arr, (oof_prob >= t).astype(int), zero_division=0)
                  for t in thresholds]
    best_thr   = thresholds[np.argmax(f1s)]
    oof_pred   = (oof_prob >= best_thr).astype(int)

    # Métricas OOF globales
    auc_oof  = roc_auc_score(y_arr, oof_prob)
    brier_oof = brier_score_loss(y_arr, oof_prob)
    f1_oof   = f1_score(y_arr, oof_pred, zero_division=0)
    sens_oof = recall_score(y_arr, oof_pred, zero_division=0)
    spec_oof = recall_score(1-y_arr, 1-oof_pred, zero_division=0)
    prec_oof = precision_score(y_arr, oof_pred, zero_division=0)
    cm       = confusion_matrix(y_arr, oof_pred)

    metricas = {
        'modelo'        : nombre,
        'n'             : int(len(y_arr)),
        'n_vars'        : int(X.shape[1]),
        'auc_oof'       : round(auc_oof, 4),
        'auc_cv_mean'   : round(np.mean(aucs), 4),
        'auc_cv_sd'     : round(np.std(aucs), 4),
        'brier'         : round(brier_oof, 4),
        'f1'            : round(f1_oof, 4),
        'sensibilidad'  : round(sens_oof, 4),
        'especificidad' : round(spec_oof, 4),
        'precision'     : round(prec_oof, 4),
        'umbral_opt'    : round(best_thr, 3),
        'TP'            : int(cm[1,1]), 'FP': int(cm[0,1]),
        'TN'            : int(cm[0,0]), 'FN': int(cm[1,0]),
    }

    # Modelo final sobre TODOS los datos (para SHAP e inferencia)
    X_all_imp, _, mice_final = mice_impute_fold(X_arr, X_arr[:1])
    scaler_final = StandardScaler()
    X_all_sc     = scaler_final.fit_transform(X_all_imp)
    clf_final    = XGBClassifier(**XGB_PARAMS)
    clf_final.fit(X_all_sc, y_arr)

    bundle = {
        'clf'     : clf_final,
        'scaler'  : scaler_final,
        'mice'    : mice_final,
        'cols'    : list(X.columns),
        'metricas': metricas,
        'umbral'  : best_thr,
    }

    return metricas, oof_prob, oof_pred, bundle

## Paso 2 — Modelos Incrementales

In [0]:
def paso_2_incrementales(df_g1, y, vars_por_fase):
    sep('PASO 2 — MODELOS INCREMENTALES')

    todos_metricas = []
    todos_oof      = {}
    bundles        = {}

    for nombre_modelo, fases_inc in MODELOS_INC.items():
        sep(f'Modelo {nombre_modelo}')

        # Acumular variables de las fases incluidas
        cols_mod = []
        for fase in fases_inc:
            cols_mod.extend(vars_por_fase.get(fase, []))
        cols_mod = list(dict.fromkeys(cols_mod))  # deduplicar

        # Controles básicos siempre presentes
        for ctrl in ['BIRTH_sexo5', 'ubicac']:
            if ctrl in df_g1.columns and ctrl not in cols_mod:
                s = to_num(df_g1[ctrl])
                if s.nunique() >= 2:
                    cols_mod.append(ctrl)

        X = df_g1[cols_mod].apply(to_num)
        logging.info(f"  Fases: {fases_inc}")
        logging.info(f"  Variables: {X.shape[1]}")
        logging.info(f"  Miss promedio: {X.isna().mean().mean()*100:.1f}%")

        metricas, oof_prob, oof_pred, bundle = evaluar_modelo(
            X, y, nombre_modelo, df_g1.index)

        todos_metricas.append(metricas)
        todos_oof[nombre_modelo] = oof_prob
        bundles[nombre_modelo]   = bundle

        logging.info(f"\n  AUC-OOF     : {metricas['auc_oof']:.4f}")
        logging.info(f"  AUC-CV μ±σ  : {metricas['auc_cv_mean']:.4f} ± {metricas['auc_cv_sd']:.4f}")
        logging.info(f"  Brier       : {metricas['brier']:.4f}")
        logging.info(f"  F1 (thr={metricas['umbral_opt']:.2f}) : {metricas['f1']:.4f}")
        logging.info(f"  Sensibilidad: {metricas['sensibilidad']:.4f}")
        logging.info(f"  Especific.  : {metricas['especificidad']:.4f}")
        logging.info(f"  CM: TP={metricas['TP']} FP={metricas['FP']} "
                     f"TN={metricas['TN']} FN={metricas['FN']}")

    df_met = pd.DataFrame(todos_metricas)
    df_met.to_csv(OUT_METRICAS, index=False, encoding='utf-8-sig')
    logging.info(f"\n  Guardado: {OUT_METRICAS}")

    return df_met, todos_oof, bundles


df_met, todos_oof, bundles = paso_2_incrementales(df_g1, y, vars_por_fase)

## Paso 3 — Interpretabilidad SHAP (Modelo M4 Completo)

In [0]:
def paso_3_shap(bundles, df_g1, y, vars_por_fase):
    sep('PASO 3 — INTERPRETABILIDAD SHAP (M4 completo)')

    bundle   = bundles['M4_completo']
    clf      = bundle['clf']
    scaler   = bundle['scaler']
    mice_imp = bundle['mice']
    cols     = bundle['cols']

    # Preparar datos para SHAP
    X_raw = df_g1[cols].apply(to_num).values.astype(float)
    X_imp = mice_imp.transform(X_raw)
    X_sc  = scaler.transform(X_imp)

    # Muestra para SHAP (máx 200 para velocidad)
    n_shap = min(200, X_sc.shape[0])
    rng    = np.random.RandomState(RANDOM_STATE)
    idx    = rng.choice(X_sc.shape[0], size=n_shap, replace=False)
    X_shap = X_sc[idx]

    explainer   = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_shap)

    # Importancias medias |SHAP|
    if isinstance(shap_values, list):
        sv = shap_values[1]   # clase positiva (GO-2 Bajo)
    else:
        sv = shap_values

    mean_shap = np.abs(sv).mean(axis=0)
    df_shap   = pd.DataFrame({
        'variable' : cols,
        'shap_mean': mean_shap.round(6),
        'shap_rank': range(1, len(cols)+1),
    }).sort_values('shap_mean', ascending=False).reset_index(drop=True)
    df_shap['shap_rank'] = range(1, len(df_shap)+1)

    df_shap.to_csv(OUT_SHAP, index=False, encoding='utf-8-sig')
    logging.info(f"  Guardado: {OUT_SHAP}")
    logging.info(f"\n  Top 15 variables por |SHAP|:")
    logging.info(f"  {'Rank':>5} {'Variable':<45} {'SHAP medio':>12}")
    logging.info('  ' + '─'*65)
    for _, r in df_shap.head(15).iterrows():
        logging.info(f"  {int(r['shap_rank']):>5} {r['variable']:<45} {r['shap_mean']:>12.5f}")

    return sv, X_shap, cols, df_shap


shap_vals, X_shap, cols, df_shap = paso_3_shap(bundles, df_g1, y, vars_por_fase)

## Paso 4 — Figuras

In [0]:
def paso_4_figuras(df_met, todos_oof, y, shap_vals, X_shap, cols, df_shap):
    sep('PASO 4 — FIGURAS')
    sns.set_theme(style='whitegrid')

    # ── fig 01: AUC incremental ───────────────────────────────────────────
    modelos = list(MODELOS_INC.keys())
    aucs    = [df_met.loc[df_met['modelo']==m, 'auc_oof'].values[0] for m in modelos]
    sds     = [df_met.loc[df_met['modelo']==m, 'auc_cv_sd'].values[0] for m in modelos]
    colores = [COLORES_MOD[m] for m in modelos]
    labels  = [LABELS[m].replace(' — ','\n') for m in modelos]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(labels, aucs, color=colores, edgecolor='white',
                  alpha=0.9, yerr=sds, capsize=5,
                  error_kw=dict(elinewidth=1.2, ecolor='#555'))
    ax.axhline(0.5, color='red', ls='--', lw=1.2, alpha=0.6, label='Azar (AUC=0.5)')
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('AUC-ROC (out-of-fold)', fontsize=11)
    ax.set_title('Modelos incrementales — AUC-ROC por fase temporal\n'
                 'Target: GO-2 Bajo vs GO-1 Alto  ·  G1 prematuros n=386',
                 fontweight='bold', fontsize=11)
    for bar, auc in zip(bars, aucs):
        ax.text(bar.get_x()+bar.get_width()/2, auc+0.01,
                f'{auc:.3f}', ha='center', va='bottom',
                fontsize=9.5, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('fig_paso_d_01_auc_incremental.png', dpi=150, bbox_inches='tight')
    plt.show()
    logging.info("  fig_paso_d_01_auc_incremental.png")

    # ── fig 02: ROC curves todos los modelos ──────────────────────────────
    y_arr = y.values
    fig, ax = plt.subplots(figsize=(7, 6))
    for m in modelos:
        prob = todos_oof[m]
        fpr, tpr, _ = roc_curve(y_arr, prob)
        auc = roc_auc_score(y_arr, prob)
        ax.plot(fpr, tpr, lw=2, color=COLORES_MOD[m],
                label=f"{LABELS[m]} (AUC={auc:.3f})")
    ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
    ax.set_xlabel('1 − Especificidad (FPR)', fontsize=11)
    ax.set_ylabel('Sensibilidad (TPR)', fontsize=11)
    ax.set_title('Curvas ROC — Modelos incrementales\n'
                 'Predicción GO-2 Bajo (fenotipo cognitivo bajo)',
                 fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig('fig_paso_d_02_roc_curvas.png', dpi=150, bbox_inches='tight')
    plt.show()
    logging.info("  fig_paso_d_02_roc_curvas.png")

    # ── fig 03: SHAP beeswarm ─────────────────────────────────────────────
    top_vars = df_shap.head(20)['variable'].tolist()
    top_idx  = [cols.index(v) for v in top_vars if v in cols]
    sv_top   = shap_vals[:, top_idx]
    X_top    = X_shap[:, top_idx]

    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(sv_top, X_top, feature_names=top_vars,
                      show=False, max_display=20, plot_type='dot',
                      color_bar_label='Valor de la variable\n(azul=bajo, rojo=alto)')
    ax = plt.gca()
    ax.set_title('SHAP — Top 20 variables predictivas (M4 completo)\n'
                 'Impacto en la predicción de GO-2 Bajo',
                 fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig('fig_paso_d_03_shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    logging.info("  fig_paso_d_03_shap_beeswarm.png")

    # ── fig 04: SHAP bar — importancia media ──────────────────────────────
    top15 = df_shap.head(15).sort_values('shap_mean')
    fig, ax = plt.subplots(figsize=(9, 6))
    colores_bar = ['#1A5276' if i < 5 else '#2E86C1' if i < 10 else '#AED6F1'
                   for i in range(len(top15))]
    ax.barh(top15['variable'], top15['shap_mean'],
            color=colores_bar[::-1], edgecolor='white', alpha=0.88)
    ax.set_xlabel('Importancia media |SHAP|', fontsize=11)
    ax.set_title('Top 15 variables — Importancia SHAP (M4 completo)\n'
                 'Predicción GO-2 Bajo vs GO-1 Alto',
                 fontweight='bold', fontsize=11)
    plt.tight_layout()
    plt.savefig('fig_paso_d_04_shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    logging.info("  fig_paso_d_04_shap_bar.png")

    # ── fig 05: Calibración M4 ────────────────────────────────────────────
    prob_m4 = todos_oof['M4_completo']
    frac_pos, mean_pred = calibration_curve(y_arr, prob_m4, n_bins=8)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Calibración perfecta')
    ax.plot(mean_pred, frac_pos, 'o-', color='#1A5276', lw=2, ms=7,
            label=f'M4 completo (Brier={brier_score_loss(y_arr, prob_m4):.3f})')
    ax.set_xlabel('Probabilidad predicha media', fontsize=11)
    ax.set_ylabel('Fracción de positivos reales', fontsize=11)
    ax.set_title('Curva de calibración — M4 completo\n'
                 'Predicción GO-2 Bajo', fontweight='bold', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('fig_paso_d_05_calibracion.png', dpi=150, bbox_inches='tight')
    plt.show()
    logging.info("  fig_paso_d_05_calibracion.png")


paso_4_figuras(df_met, todos_oof, y, shap_vals, X_shap, cols, df_shap)

## Paso 5 — Resumen y Export

In [0]:
def paso_5_resumen(df_met, bundles):
    sep('PASO 5 — RESUMEN EJECUTIVO')

    logging.info(f"\n  {'Modelo':<30} {'n_vars':>7}  {'AUC':>7}  {'F1':>7}  "
                 f"{'Sens':>7}  {'Spec':>7}  {'Brier':>7}")
    logging.info('  ' + '─'*78)
    for _, r in df_met.iterrows():
        logging.info(f"  {r['modelo']:<30} {r['n_vars']:>7}  {r['auc_oof']:>7.4f}  "
                     f"{r['f1']:>7.4f}  {r['sensibilidad']:>7.4f}  "
                     f"{r['especificidad']:>7.4f}  {r['brier']:>7.4f}")

    best = df_met.loc[df_met['auc_oof'].idxmax()]
    logging.info(f"\n  Mejor modelo: {best['modelo']} (AUC={best['auc_oof']:.4f})")

    # Guardar M4 serializado
    joblib.dump(bundles['M4_completo'], OUT_MODELO)
    logging.info(f"  Modelo M4 guardado: {OUT_MODELO}")

    sep('OUTPUTS GENERADOS')
    for f in [OUT_METRICAS, OUT_SHAP, OUT_MODELO, LOG_FILE,
              'fig_paso_d_01_auc_incremental.png',
              'fig_paso_d_02_roc_curvas.png',
              'fig_paso_d_03_shap_beeswarm.png',
              'fig_paso_d_04_shap_bar.png',
              'fig_paso_d_05_calibracion.png']:
        estado = 'OK' if os.path.exists(f) else 'pendiente'
        logging.info(f"  {estado}  {f}")


paso_5_resumen(df_met, bundles)